# Credit Portfolio & Customer Profitability Analytics

**Business case:** Analyze a retail-bank lending portfolio to identify profitable products, risky customer cohorts, regional performance gaps, and priority accounts for early intervention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
df = pd.read_csv('../data/credit_portfolio.csv')
df.head()

## 1. Executive KPIs

In [ ]:
kpis = {
    "Accounts": len(df),
    "Outstanding Balance": df.outstanding_balance.sum(),
    "Default Rate %": df.default_flag.mean()*100,
    "Average CIBIL": df.cibil_score.mean(),
    "Net Monthly Revenue": df.net_monthly_revenue.sum()
}
pd.Series(kpis).round(2)

## 2. Product profitability vs risk

In [ ]:
product = df.groupby('product').agg(
    accounts=('customer_id','count'),
    outstanding=('outstanding_balance','sum'),
    default_rate=('default_flag','mean'),
    net_revenue=('net_monthly_revenue','sum')
).sort_values('net_revenue', ascending=False)

product.assign(default_rate_pct=product.default_rate*100).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
product.net_revenue.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Net Monthly Revenue by Product')
ax.set_xlabel('Net Revenue')
plt.tight_layout()

## 3. Regional performance

In [ ]:
region = df.groupby('region').agg(
    accounts=('customer_id','count'),
    outstanding=('outstanding_balance','sum'),
    default_rate=('default_flag','mean'),
    net_revenue=('net_monthly_revenue','sum')
)
region.assign(default_rate_pct=region.default_rate*100).round(2).sort_values('net_revenue', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
(region.default_rate*100).sort_values().plot(kind='bar', ax=ax)
ax.set_title('Default Rate by Region')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()

## 4. Risk concentration

In [ ]:
risk = df.groupby('risk_band').agg(
    accounts=('customer_id','count'),
    outstanding=('outstanding_balance','sum'),
    default_rate=('default_flag','mean'),
    expected_loss=('expected_credit_loss','sum')
)
risk.assign(default_rate_pct=risk.default_rate*100).round(2).sort_values('outstanding', ascending=False)

## 5. Early-warning customer list

Flag accounts with low CIBIL, 30+ DPD, or high credit utilization.

In [ ]:
early_warning = df[
    (df.cibil_score < 650) |
    (df.dpd_days >= 30) |
    (df.credit_utilization_pct >= 85)
].sort_values('outstanding_balance', ascending=False)

early_warning[['customer_id','product','branch_id','cibil_score',
               'dpd_days','credit_utilization_pct','outstanding_balance']].head(20)

## 6. Business recommendations

1. Prioritize high-outstanding accounts with CIBIL <650 or 30+ DPD for early collections.
2. Compare product profitability with default rate before pushing volume-led growth.
3. Focus cross-sell on affluent/low-risk cohorts with strong repayment behavior.
4. Use branch/region dashboards to identify high-revenue but high-risk pockets.
5. Monitor EMI-to-income and credit utilization as early-warning indicators.